# 02 - RF-DETR Nano (package rfdetr de Roboflow)

Dataset COCO 'plat' (train/valid/test) construit depuis les memes listes de tuiles. Batch 8 x 4 = 32 effectif. Augmentation : `augment.AUG_RFDETR`.

In [ ]:
# --- Installation ---
!pip install -q "rfdetr[train,loggers]" pycocotools wandb

In [ ]:
# --- Connexion Drive ---
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# --- Code du benchmark (package aphids_det) ---
import os, sys
REPO_DIR = "/content/aphids_detection"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/EmmaDub/aphids_detection.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
sys.path.insert(0, REPO_DIR)
import aphids_det
print("aphids_det", aphids_det.__version__)

In [ ]:
# --- CONFIG DONNEES (source des tuiles + variante labels_cell_20) ---
# Memes chemins que comparaison_modeles_ultralytics.ipynb : c'est ici, et nulle
# part ailleurs, qu'on change de jeu de donnees.
from pathlib import Path
import aphids_det.config as cfg

cfg.BASE_DIR = Path("/content/drive/MyDrive/Emma/puceron_model_2026/"
                    "puceron_model_E2026_3/data/tuile_viz02_640_128")
cfg.SPLIT_DIR = cfg.BASE_DIR / "split"
cfg.CELL_DIR = Path("/content/drive/MyDrive/Emma/puceron_model_2026/"
                    "puceron_model_E2026_2/data/cell/tuile_viz02_640_128_cell")
cfg.MAIN_IMAGES_DIR = cfg.BASE_DIR / "images"
cfg.BG_DIR = Path("/content/drive/MyDrive/Emma/puceron_model_2026/"
                  "puceron_model_E2026_2/data/images_complete")
cfg.BG_ZIP = cfg.BG_DIR.parent / "tuile_viz02_640_128_background.zip"

cfg.EXPERIMENT_NAME = "yolo_neg1"
cfg.SEARCH_VARIANT = "labels_cell_20"
cfg.VARIANTS = {
    "labels_cell_20": cfg.V(
        cfg.SPLIT_DIR / "split_assignments_all_background.csv", "labels_visible_20",
        extra_train={"csv":        cfg.CELL_DIR / "split" / "split_assignments.csv",
                     "labels_dir": cfg.CELL_DIR / "labels_cell_20",
                     "images_dir": cfg.CELL_DIR / "images_lookmatched3"}),
}

cfg.CLASS_NAMES = {0: "Apterous_aphid", 1: "Alate_aphid"}
cfg.N_CV_FOLDS = 5          # folds 0-4 en validation croisee
cfg.TEST_FOLD = 5           # fold 5 : test, jamais utilise ici
cfg.CV_FOLDS = list(range(cfg.N_CV_FOLDS))
cfg.NEG_RATIO = 3

# --- SORTIES (Drive) et budget ---
cfg.OUT_DIR = Path("/content/drive/MyDrive/Emma/puceron_model_2026/"
                   "puceron_model_article/data")
cfg.EPOCHS = 30
cfg.IMGSZ = 640
cfg.PATIENCE = 5
cfg.SEED = 42
cfg.USE_WANDB = True
cfg.WANDB_PROJECT = "comparaison_pucerons_detection"

cfg.refresh()
cfg.summary()

In [ ]:
# --- Verification des chemins Drive avant de lancer quoi que ce soit ---
attendus = {
    "tuiles (images)": cfg.MAIN_IMAGES_DIR,
    "labels": cfg.VARIANTS[cfg.SEARCH_VARIANT]["labels_dir"],
    "CSV de split": cfg.VARIANTS[cfg.SEARCH_VARIANT]["csv"],
    "fonds (images_complete)": cfg.BG_DIR,
    "renfort cell : images": cfg.VARIANTS[cfg.SEARCH_VARIANT]["extra_train"]["images_dir"],
    "renfort cell : labels": cfg.VARIANTS[cfg.SEARCH_VARIANT]["extra_train"]["labels_dir"],
    "renfort cell : CSV": cfg.VARIANTS[cfg.SEARCH_VARIANT]["extra_train"]["csv"],
    "sorties (Drive)": cfg.OUT_DIR,
}
for nom, p in attendus.items():
    p = Path(p)
    etat = "OK     " if p.exists() else "MANQUE "
    extra = ""
    if p.is_dir():
        try:
            extra = f"  ({sum(1 for _ in p.iterdir())} entrees)"
        except OSError:
            pass
    print(f"{etat}{nom:28s} {p}{extra}")
if not Path(cfg.BG_DIR).exists():
    print(f"\n(fonds absents : l'archive {cfg.BG_ZIP} sera extraite en local)")

In [ ]:
# --- Construction des folds (symlinks locaux, a refaire a chaque session Colab) ---
from aphids_det import folds
folds.build_folds()

In [ ]:
# --- Suivi W&B (facultatif : mettre cfg.USE_WANDB = False pour s'en passer) ---
if cfg.USE_WANDB:
    import wandb
    wandb.login()
    os.environ["WANDB_PROJECT"] = cfg.WANDB_PROJECT
    print("W&B -> projet", cfg.WANDB_PROJECT)

In [ ]:
from aphids_det.runners import rfdetr_runner
df = rfdetr_runner.run_cv()

In [ ]:
# --- Etat du CSV de benchmark ---
import pandas as pd
d = pd.read_csv(cfg.CSV_CV)
print(d.groupby("modele")["fold"].count().to_string(), "\n")
d[["modele", "fold", "map50_macro", "map5095_macro", "latency_cpu_ms",
   "n_params_M", "train_time_s"]].tail(10)